# 05 · Lightweight industrial instance segmentation
Optional: skip this notebook when using the pretrained bundle from notebook 04a.
Train YOLO11n-seg on real masks. GPU recommended; CPU is supported but slow.
This notebook stops if notebook 04's validated dataset is missing.
Hue-changing augmentation is disabled to preserve red/green/blue semantics.
MobileNetV3 visible-quality training runs only with criteria and all labels in all splits.
Original geometry heuristics are not defect labels for nuts/washers/brackets.
Export includes CPU ONNX, original weights, classes, thresholds and runtime source.
Ultralytics licensing applies; review it before distributing a closed product.


In [ ]:
# Change these values BEFORE running subsequent cells.
USE_DRIVE = True
ROOT = '/content/drive/MyDrive/Robotekin2026'
SEED = 42
DATASET_PATH = ''  # Existing dataset folder on Drive; empty opens upload.
MODEL_PATH = ''    # Existing exported model folder; empty opens upload.
RESUME_RUN = ''    # Existing run folder for checkpoint continuation.
EPOCHS = 80
IMGSZ = 640
TRAIN_QUALITY = True
QUALITY_EPOCHS = 20


In [ ]:
import os, random, subprocess, sys, uuid
from datetime import datetime, timezone
from pathlib import Path
print('Python:', sys.version, '| executable:', sys.executable, flush=True)
install_log = Path('/content/roboqc_pip_install.log')
install_log.parent.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', 'numpy>=1.26,<3', 'scikit-learn>=1.4,<2', 'joblib>=1.3,<2', 'matplotlib>=3.8,<4', 'pandas>=2.2,<4', 'psutil>=5.9,<8', 'torch>=2.5,<3', 'torchvision>=0.20,<1', 'ultralytics>=8.4.0,<9', 'opencv-python-headless>=4.10,<6', 'pycocotools>=2.0.8,<3', 'PyYAML>=6,<7', 'onnx>=1.17,<2', 'onnxruntime>=1.20,<2']
with install_log.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
    exit_code = process.wait()
if exit_code:
    raise RuntimeError(f'pip failed (exit {exit_code}). Read the ERROR lines above. Full log: {install_log}')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    ROOT = '/content/Robotekin2026'
RUN = Path(RESUME_RUN) if RESUME_RUN else Path(ROOT) / (datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6])
RUN.mkdir(parents=True, exist_ok=bool(RESUME_RUN))
import numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
GPU = torch.cuda.is_available()
DEVICE = 'cuda' if GPU else 'cpu'
print('Device:', DEVICE, '| Run:', RUN)
(RUN / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


In [ ]:
import sys, json, zipfile, shutil, uuid
from pathlib import Path
RUNTIME = Path('/content/roboqc_runtime')
SOURCES = {'roboqc/__init__.py': '', 'roboqc/intent/__init__.py': '', 'roboqc/industrial/__init__.py': '"""Industrial sorting contracts shared by Colab and the local demo."""\n', 'roboqc/industrial/voice.py': '"""Conservative Spanish sorting parser and explicit clarification dialogue.\n\nNo model can bypass relation validation. No robot or GUI dependencies.\n"""\n\nfrom __future__ import annotations\n\nimport copy\nimport re\nimport unicodedata\n\nCOLORS = ("red", "green", "blue")\nCATEGORIES = {"quality": ("good", "defect_1", "defect_2"), "color": COLORS}\nCOLOR_WORDS = {"red": "roja", "green": "verde", "blue": "azul"}\nQUALITY = {\n    "good": r"\\b(?:buen[oa]s?|correct[oa]s?|aceptables?|sin defectos?)\\b",\n    "defect_1": r"\\b(?:defect(?:uos[oa]s?|os?)\\s*(?:de\\s+)?(?:tipo\\s*)?(?:uno|1)|tipo\\s+(?:uno|1))\\b",\n    "defect_2": r"\\b(?:defect(?:uos[oa]s?|os?)\\s*(?:de\\s+)?(?:tipo\\s*)?(?:dos|2)|tipo\\s+(?:dos|2))\\b",\n}\nCOLOR_PATTERNS = {"red": r"\\broj[oa]s?\\b", "green": r"\\bverdes?\\b", "blue": r"\\bazul(?:es)?\\b"}\nDEST = re.compile(r"\\b(?:en|a|al|hacia|dentro de)\\s+(?:(?:la|el|las|los|una?|del)\\s+)?(?:(?:cajas?|contenedor(?:es)?|recipientes?|destinos?)\\s+)?(roj[oa]s?|verdes?|azul(?:es)?|amarill[oa]s?|negr[oa]s?|blanc[oa]s?)\\b")\nSORT = r"\\b(?:clasifica\\w*|ordena\\w*|separa\\w*|distribu\\w*|reparte\\w*|pon|poner|coloca\\w*|deja\\w*|lleva\\w*|envia\\w*|manda\\w*|recoge\\w*|recoger|coge\\w*|recoger|organiza\\w*)\\b"\n\n\ndef normalize(text):\n    return " ".join("".join(c for c in unicodedata.normalize("NFKD", text.casefold()) if not unicodedata.combining(c)).split())\n\n\ndef handles(text):\n    t = normalize(text)\n    if re.match(r"^(?:cuant|donde|muestra|selecciona|inspecciona|hay |que )", t.lstrip("¿")):\n        return False\n    return bool(re.search(r"\\b(?:clasifica\\w*|continua|reanuda|deten|cancela|anula|para|pausa|interrumpe|sigue)\\b", t)\n                or (re.search(SORT, t) and re.search(r"\\b(?:las|piezas|tornillos|tuercas|arandelas|soportes)\\b", t)))\n\n\ndef mentions(text, patterns):\n    return [(m.start(), m.end(), key) for key, pat in patterns.items() for m in re.finditer(pat, text)]\n\n\ndef result(command=None, status="valid", issues=None, clarification=None):\n    return {"schema_version": "industrial-1.0", "status": status, "command": command,\n            "issues": issues or [], "clarification": clarification}\n\n\ndef canonical(command):\n    """Ordering-independent semantic comparison; excludes latency/confidence."""\n    value = copy.deepcopy(command)\n    if value and "rules" in value:\n        value["rules"] = sorted(value["rules"], key=lambda r: (r["category"], r.get("destination") or ""))\n    return value\n\n\ndef understand(text, model=None):\n    t = normalize(text)\n    if not t:\n        return result(status="invalid", issues=["empty_text"])\n    if len(t) > 2000:\n        return result(status="unsupported", issues=["command_too_long"])\n    if re.search(r"(?<!\\w)-\\d+\\b", t):\n        return result(status="invalid", issues=["negative_quantity"])\n    if re.search(r"\\b(?:suelda\\w*|pinta\\w*|taladra\\w*|pule|atornilla\\w*|empaqueta\\w*|desmonta\\w*)\\b", t):\n        return result(status="unsupported", issues=["unsupported_action"])\n    # Match full control phrases: a negated stop must never become a stop.\n    clean = re.sub(r"^(?:por favor[, ]+|robot[, ]+)", "", t).strip(" .!?¿¡")\n    for intent, pattern in {\n        "stop": r"(?:deten|para|interrumpe|pausa)(?:\\s+(?:la\\s+)?(?:clasificacion|proceso|robot))?(?:\\s+por favor)?",\n        "resume": r"(?:continua|reanuda|sigue)(?:\\s+(?:con\\s+)?(?:la\\s+)?(?:clasificacion|clasificando|proceso))?(?:\\s+por favor)?",\n        "cancel": r"(?:cancela|anula)(?:\\s+(?:la\\s+)?(?:orden|peticion|clasificacion))?",\n    }.items():\n        if re.fullmatch(pattern, clean):\n            return result({"intent": intent})\n    if re.search(r"\\b(?:no|nunca)\\b", t) and re.search(r"\\b(?:detengas|pares|continues|reanudes)\\b", t):\n        return result(status="unsupported", issues=["negated_control"])\n    if re.search(r"\\b(?:deten|pausa|continua|reanuda|cancela)\\b|\\bpara\\s+(?:el\\s+(?:proceso|robot)|la\\s+clasificacion)\\b", t) and re.search(SORT, t):\n        return result(status="contradictory", issues=["mixed_control_and_sort"])\n    q = mentions(t, QUALITY)\n    predicted = model.predict(t) if model is not None else None\n    # A learned action can extend verb coverage, but never supply missing destinations.\n    is_sort = bool(re.search(SORT, t)) or bool(predicted and predicted.get("intent") == "sort")\n    if not is_sort:\n        return result(status="unsupported", issues=["unrecognized_action"])\n    if re.search(r"\\b(?:excepto|menos|salvo|no|nunca)\\b", t):\n        return result(status="needs_clarification", issues=["negation_scope"],\n                      clarification="Indica las categorias que quieres clasificar y sus destinos de forma positiva.")\n    destinations = list(DEST.finditer(t))\n    source_text = DEST.sub(" ", t)\n    c = mentions(source_text, COLOR_PATTERNS)\n    if q and c:\n        return result(status="needs_clarification", issues=["mixed_modes"],\n                      clarification="¿Quieres clasificar por calidad o por color?")\n    mode = "quality" if q or re.search(r"\\b(?:defectuosas?|defectuosos?|calidad)\\b", t) else "color"\n    patterns = QUALITY if mode == "quality" else COLOR_PATTERNS\n    rules = []\n    cursor = 0\n    for dest in destinations:\n        clause = t[cursor:dest.start()]\n        categories = [item[2] for item in sorted(mentions(clause, patterns))]\n        consumed_end = dest.end()\n        if not categories:\n            after = re.split(r"\\s+y\\s+|[,;]", t[dest.end():], maxsplit=1)[0]\n            categories = [item[2] for item in sorted(mentions(after, patterns))]\n            if categories:\n                consumed_end += len(after)\n        if mode == "quality" and not categories and re.search(r"\\bdefectuos[oa]s?\\b", clause):\n            categories = ["defect_1", "defect_2"]\n        color = next((key for key, pat in COLOR_PATTERNS.items() if re.search(pat, dest.group(1))), None)\n        if color is None:\n            return result(status="unsupported", issues=["unsupported_destination"])\n        if not categories:\n            return result(status="needs_clarification", issues=["missing_category"],\n                          clarification="¿Que categoria va en la caja indicada?")\n        rules.extend({"category": category, "destination": color} for category in categories)\n        cursor = consumed_end\n    trailing = [item[2] for item in mentions(t[cursor:], patterns)]\n    if destinations and re.search(r"\\b(?:en|hacia|caja|contenedor)\\b", t[cursor:]):\n        return result(status="needs_clarification", issues=["unresolved_trailing_clause"],\n                      clarification="Repite cada categoria con un unico destino completo.")\n    if not destinations:\n        trailing = [item[2] for item in mentions(t, patterns)]\n        if mode == "quality" and not trailing and re.search(r"\\bdefectuos[oa]s?\\b", t):\n            trailing = ["defect_1", "defect_2"]\n    rules.extend({"category": category, "destination": None} for category in trailing)\n    command = {"intent": "sort", "mode": mode, "rules": rules}\n    if re.search(r"\\b(?:solamente|solo|unicamente)\\b", t):\n        command["filter"] = list(dict.fromkeys(r["category"] for r in rules))\n    return validate(command)\n\n\ndef validate(command):\n    command = copy.deepcopy(command)\n    mode = command.get("mode")\n    if mode not in CATEGORIES:\n        return result(status="invalid", issues=["invalid_mode"])\n    unique = {}\n    for rule in command.get("rules", []):\n        cat, dest = rule.get("category"), rule.get("destination")\n        if cat not in CATEGORIES[mode] or (dest is not None and dest not in COLORS):\n            return result(status="unsupported", issues=["unsupported_rule"])\n        if cat in unique and unique[cat] != dest:\n            return result(command, "contradictory", ["conflicting_destinations"])\n        unique[cat] = dest\n    command["rules"] = [{"category": c, "destination": d} for c, d in unique.items()]\n    if not unique:\n        return result(command, "needs_clarification", ["missing_category"], "¿Que categorias y destinos quieres usar?")\n    if any(d is None for d in unique.values()):\n        missing = ", ".join(c for c, d in unique.items() if d is None)\n        return result(command, "needs_clarification", ["missing_destination"],\n                      f"¿A que caja (roja, verde o azul) envio {missing}?")\n    return result(command)\n\n\nclass Dialogue:\n    """A missing destination always needs an explicit answer, even with active rules."""\n\n    def __init__(self):\n        self.pending = None\n        self.active = None\n        self.running = False\n\n    def submit(self, text, model=None):\n        t = normalize(text)\n        if self.pending and not re.search(SORT, t) and not re.search(r"\\b(?:deten|para|pausa|cancela|anula|continua|reanuda)\\b", t):\n            pending_command = self.pending.get("command")\n            if pending_command and "missing_destination" in self.pending["issues"]:\n                candidate = copy.deepcopy(pending_command)\n                explicit = understand("pon " + text)\n                if explicit["status"] == "valid" and explicit.get("command", {}).get("rules"):\n                    updates = {r["category"]: r["destination"] for r in explicit["command"]["rules"]}\n                    for rule in candidate["rules"]:\n                        if rule["destination"] is None and rule["category"] in updates:\n                            rule["destination"] = updates[rule["category"]]\n                else:\n                    colors = {m[2] for m in mentions(t, COLOR_PATTERNS)}\n                    # Only a bare destination answer can apply one color to all missing rules.\n                    if len(colors) == 1 and re.fullmatch(r"(?:(?:a|en)\\s+)?(?:(?:la|el)\\s+)?(?:(?:caja|contenedor)\\s+)?(?:roja?|rojo|verde|azul)[.! ]*", t):\n                        for rule in candidate["rules"]:\n                            if rule["destination"] is None:\n                                rule["destination"] = next(iter(colors))\n                response = validate(candidate)\n            else:\n                response = result(status="needs_clarification", issues=["repeat_complete_command"],\n                                  clarification="Reformula la instruccion completa con categorias y destinos.")\n        else:\n            response = understand(text, model)\n        if response["status"] == "needs_clarification":\n            self.pending = response\n            self.running = False\n        elif response["status"] == "valid":\n            intent = response["command"]["intent"]\n            if intent == "sort":\n                self.active = response["command"]\n                self.pending = None\n                self.running = True\n            elif intent == "cancel":\n                self.pending = None\n                self.active = None\n                self.running = False\n            elif intent == "stop":\n                self.running = False\n            elif intent == "resume":\n                if self.active is None or self.pending:\n                    return result(status="needs_clarification", issues=["no_resumable_plan"],\n                                  clarification="Completa una orden de clasificacion antes de continuar.")\n                self.running = True\n        else:\n            self.running = False\n        return response\n\n\ndef to_legacy(envelope, text, source="text"):\n    from roboqc.commands.schema import Action, StructuredCommand\n    status = {"valid": "VALID", "needs_clarification": "AMBIGUOUS", "unsupported": "UNKNOWN"}.get(envelope["status"], "INVALID")\n    cmd = StructuredCommand(source=source, transcription=text, normalized_text=normalize(text), status=status)\n    cmd.actions = [Action("INDUSTRIAL_COMMAND", {"industrial": envelope})]\n    cmd.validation_errors = list(envelope["issues"])\n    if envelope["clarification"]:\n        cmd.unresolved_references = [envelope["clarification"]]\n    return cmd\n', 'roboqc/industrial/dataset.py': '"""Semantic-first synthetic commands. Called manually from Colab, never on import."""\n\nimport hashlib\nimport itertools\nimport json\nimport random\nfrom pathlib import Path\n\nfrom .voice import CATEGORIES, COLOR_WORDS, normalize, result\n\nQUALITY_WORDS = {\n    "good": ["buenas", "correctas", "aceptables", "sin defectos"],\n    "defect_1": ["defectuosas tipo uno", "con defecto tipo 1", "tipo uno", "defectuosas de tipo uno"],\n    "defect_2": ["defectuosas tipo dos", "con defecto tipo 2", "tipo dos", "defectuosas de tipo dos"],\n}\nPARTS = ["piezas", "tuercas", "arandelas", "tornillos", "soportes", "componentes"]\nVERBS = ["Pon", "Coloca", "Clasifica", "Lleva", "Envia", "Distribuye", "Manda", "Deja", "Separa", "Reparte"]\n# Entire grammatical families belong to one split; variants never cross it.\nFAMILIES = {\n    "train": ["{v} {s} en la caja {d}", "{v} {s} al contenedor {d}", "{v} {s} hacia el recipiente {d}", "{v} {s} dentro de la caja {d}"],\n    "validation": ["{v} a la caja {d}: {s}", "{v} {s} a la {d}"],\n    "test": ["{v} {s} en el destino {d}", "{v} hacia la caja {d} {s}"],\n}\nPREFIXES = ["", "Por favor, ", "Robot, ", "Ahora, ", "Cuando puedas, ", "Para esta tanda, "]\nSUFFIXES = ["", ", por favor", " con cuidado", " despacio", " ahora", " para esta tanda"]\n\n\ndef key(text):\n    return " ".join(__import__("re").sub(r"[^\\w\\s]", " ", normalize(text)).split())\n\n\ndef _render(family, verb, subject, destination):\n    color = "rojo" if destination == "red" and any(word in family for word in ("contenedor", "recipiente", "destino")) else COLOR_WORDS[destination]\n    return family.format(v=verb, s=subject, d=color)\n\n\ndef _subject(category, mode, rng):\n    description = rng.choice(QUALITY_WORDS[category]) if mode == "quality" else {"red":"rojas","green":"verdes","blue":"azules"}[category]\n    part = rng.choice(PARTS)\n    masculine = part in ("tornillos","soportes","componentes")\n    if masculine:\n        for feminine,male in [("buenas","buenos"),("correctas","correctos"),("defectuosas","defectuosos"),("rojas","rojos")]:\n            description = description.replace(feminine,male)\n    return ("los " if masculine else "las ") + part + " " + description\n\n\ndef generate(n=25000, seed=42):\n    if n < 20000:\n        raise ValueError("The voice dataset requires at least 20,000 examples.")\n    rng = random.Random(seed)\n    splits, seen = {}, set()\n    for split, count in zip(FAMILIES, [int(n * .70), int(n * .15), n - int(n * .70) - int(n * .15)]):\n        records = []\n        attempts = 0\n        while len(records) < count:\n            attempts += 1\n            if attempts > count * 300:\n                raise RuntimeError("Unique example space exhausted; expand grammatical families.")\n            mode = rng.choice(list(CATEGORIES))\n            cats = rng.sample(list(CATEGORIES[mode]), rng.randint(1, 3))\n            family_index = rng.randrange(len(FAMILIES[split]))\n            family = FAMILIES[split][family_index]\n            kind = rng.choices(["valid", "missing", "conflict", "unsupported", "control", "invalid"], [.52, .16, .10, .08, .08, .06])[0]\n            clauses, rules = [], []\n            for i, cat in enumerate(cats):\n                subject = _subject(cat,mode,rng)\n                dest = rng.choice(list(COLOR_WORDS))\n                clauses.append(_render(family, rng.choice(VERBS), subject, dest))\n                rules.append({"category": cat, "destination": dest})\n            text = rng.choice(PREFIXES) + " y ".join(clauses) + rng.choice(SUFFIXES)\n            command = {"intent": "sort", "mode": mode, "rules": rules}\n            expected = result(command)\n            if kind == "invalid":\n                text = rng.choice(PREFIXES) + _render(family,rng.choice(VERBS),f"-{rng.randint(1, 99)} piezas buenas",rng.choice(list(COLOR_WORDS)))\n                expected = result(status="invalid", issues=["negative_quantity"])\n            elif kind == "missing":\n                # Generate independently labelled missing-destination instructions.\n                cat = cats[0]\n                subject = _subject(cat,mode,rng)\n                missing_families = {"train":["{v} solamente {s}","{v} solo {s}"],\n                    "validation":["{v} unicamente {s} para esta tanda"],\n                    "test":["{v} para esta tanda solamente {s}"]}\n                text = rng.choice(PREFIXES) + rng.choice(missing_families[split]).format(v=rng.choice(VERBS),s=subject) + rng.choice(SUFFIXES)\n                command = {"intent": "sort", "mode": mode, "rules": [{"category": cat, "destination": None}], "filter": [cat]}\n                expected = result(command, "needs_clarification", ["missing_destination"])\n            elif kind == "conflict":\n                cat = cats[0]\n                other = rng.choice([d for d in COLOR_WORDS if d != rules[0]["destination"]])\n                desc = rng.choice(QUALITY_WORDS[cat]) if mode == "quality" else {"red": "rojas", "green": "verdes", "blue": "azules"}[cat]\n                text += " y " + _render(family, rng.choice(VERBS), "las piezas " + desc, other)\n                command["rules"].append({"category": cat, "destination": other})\n                expected = result(command, "contradictory", ["conflicting_destinations"])\n            elif kind == "unsupported":\n                text = rng.choice(PREFIXES) + _render(family,rng.choice(["Suelda", "Pinta", "Taladra", "Pule", "Atornilla", "Empaqueta"]),_subject(cats[0],mode,rng),rng.choice(list(COLOR_WORDS))) + rng.choice(SUFFIXES)\n                expected = result(status="unsupported", issues=["unrecognized_action"])\n            elif kind == "control":\n                intent = rng.choice(["stop", "resume", "cancel"])\n                phrase = {\n                    "train": {"stop": ["Deten la clasificacion"], "resume": ["Continua clasificando"], "cancel": ["Cancela la orden"]},\n                    "validation": {"stop": ["Interrumpe el proceso"], "resume": ["Reanuda la clasificacion"], "cancel": ["Anula la peticion"]},\n                    "test": {"stop": ["Para el robot"], "resume": ["Sigue con la clasificacion"], "cancel": ["Cancela la clasificacion"]},\n                }[split]\n                text = rng.choice(["", "Por favor, ", "Robot, "]) + rng.choice(phrase[intent]) + rng.choice(["", ".", "!"])\n                expected = result({"intent": intent})\n            norm = key(text)\n            if norm in seen:\n                continue\n            seen.add(norm)\n            group = f"{split}:{family_index}:{kind}"\n            record = {"id": hashlib.sha256(norm.encode()).hexdigest()[:16], "text": text,\n                      "family": f"{split}:{family_index}", "group": group, "split": split,\n                      "expected": expected, "intent": (expected["command"] or {}).get("intent", "unsupported"),\n                      "mode": (expected["command"] or {}).get("mode", "none")}\n            records.append(record)\n        splits[split] = records\n    return splits\n\n\ndef hypotheses():\n    return [f"La orden pide {action}." for action in ("clasificar piezas", "detener la clasificacion", "continuar clasificando", "cancelar la orden")]\n\n\ndef pairs(record):\n    """Relations and action decisions; neutral is distinct from contradiction."""\n    command = record["expected"]["command"] or {}\n    for action, hypothesis in zip(["sort", "stop", "resume", "cancel"], hypotheses()):\n        yield record["text"], hypothesis, "entailment" if command.get("intent") == action else "neutral"\n    for mode in CATEGORIES:\n        yield record["text"], f"La orden pide clasificar por {\'calidad\' if mode == \'quality\' else \'color\'}.", "entailment" if command.get("mode") == mode else "neutral"\n    yield record["text"], "La orden solicita una accion no admitida.", "entailment" if record["intent"] == "unsupported" else "neutral"\n    for cat, dest in itertools.product(CATEGORIES.get(command.get("mode"), []), COLOR_WORDS):\n        matches = [r for r in command.get("rules", []) if r["category"] == cat]\n        label = "entailment" if any(r["destination"] == dest for r in matches) else "contradiction" if any(r["destination"] for r in matches) else "neutral"\n        yield record["text"], f"El destino de {cat} es la caja {dest}.", label\n\n\ndef export(splits, output):\n    import csv\n    from collections import Counter\n    out = Path(output)\n    out.mkdir(parents=True, exist_ok=False)\n    keys, families, report = {}, {}, {}\n    for split, records in splits.items():\n        keys[split] = {key(r["text"]) for r in records}\n        families[split] = {r["family"] for r in records}\n        assert len(keys[split]) == len(records)\n        for record in records:\n            json.dumps(record["expected"], ensure_ascii=False, allow_nan=False)\n        with (out / f"{split}.jsonl").open("w", encoding="utf-8") as f:\n            for record in records:\n                f.write(json.dumps(record, ensure_ascii=False) + "\\n")\n        with (out / f"{split}.csv").open("w", encoding="utf-8", newline="") as f:\n            writer = csv.DictWriter(f, fieldnames=list(records[0]))\n            writer.writeheader()\n            writer.writerows({**r, "expected": json.dumps(r["expected"], ensure_ascii=False)} for r in records)\n        report[split] = {"size": len(records), "status": dict(Counter(r["expected"]["status"] for r in records)),\n                         "intent": dict(Counter(r["intent"] for r in records))}\n    for a, b in itertools.combinations(keys, 2):\n        assert not keys[a] & keys[b], (a, b, "duplicate text")\n        assert not families[a] & families[b], (a, b, "family leakage")\n    (out / "manifest.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    return report\n', 'roboqc/industrial/models.py': '"""Portable intent bundles. Heavy backends are imported only when selected."""\n\nimport json\nfrom pathlib import Path\n\nLABELS = ["sort|quality", "sort|color", "stop|none", "resume|none", "cancel|none", "unsupported|none"]\n\n\ndef decode(label, confidence=1.0):\n    intent, mode = label.split("|")\n    return {"intent": intent, "mode": mode, "confidence": float(confidence)}\n\n\nclass ONNXSentenceEncoder:\n    """Sentence embeddings through ONNX Runtime, independent of Optimum."""\n\n    def __init__(self, path):\n        import onnxruntime as ort\n        from transformers import AutoTokenizer\n        path = Path(path)\n        self.config = json.loads((path / "onnx_config.json").read_text(encoding="utf-8"))\n        self.tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)\n        self.session = ort.InferenceSession(str(path / self.config["model_file"]), providers=["CPUExecutionProvider"])\n\n    def encode(self, texts, show_progress_bar=False, batch_size=32):\n        import numpy as np\n        outputs = []\n        for start in range(0,len(texts),batch_size):\n            tokens = self.tokenizer(texts[start:start+batch_size],padding=True,truncation=True,\n                                    max_length=self.config["max_seq_length"],return_tensors="np")\n            inputs = {item.name: tokens[item.name].astype(np.int64) for item in self.session.get_inputs()}\n            outputs.append(self.session.run(["sentence_embedding"],inputs)[0])\n        return np.concatenate(outputs) if outputs else np.empty((0,self.config["embedding_dim"]),dtype=np.float32)\n\n\nclass IntentBundle:\n    def __init__(self, path, kind=None):\n        self.root = Path(path)\n        self.manifest = json.loads((self.root / "manifest.json").read_text(encoding="utf-8"))\n        if self.manifest.get("schema_version") != "industrial-1.0":\n            raise ValueError("Unsupported industrial artifact version")\n        self.kind = kind or self.manifest["selected"]\n        if self.kind not in self.manifest["models"]:\n            raise ValueError(f"Model {self.kind} was not exported")\n        self.threshold = self.manifest["models"][self.kind].get("threshold", 0.0)\n        self._cache = {}\n        if self.kind == "tfidf":\n            import joblib\n            self.model = joblib.load(self.root / "tfidf.joblib")\n        elif self.kind == "minilm":\n            import joblib\n            self.model = joblib.load(self.root / "minilm.joblib")\n            info = self.manifest["models"][self.kind]\n            path = self.root / info.get("encoder_path", "encoder")\n            if info.get("encoder_backend") == "onnx_native":\n                self.encoder = ONNXSentenceEncoder(path)\n            else:\n                from sentence_transformers import SentenceTransformer\n                kwargs = {"backend": info.get("encoder_backend", "torch")}\n                if info.get("encoder_file"):\n                    kwargs["model_kwargs"] = {"file_name": info["encoder_file"]}\n                self.encoder = SentenceTransformer(str(path), device="cpu", **kwargs)\n        elif self.kind == "jepa":\n            import torch\n            from roboqc.intent.jepa_model import CharVocab, JEPAIntentModel\n            ckpt = torch.load(self.root / "jepa.pt", map_location="cpu", weights_only=True)\n            self.vocab = CharVocab.from_dict(ckpt["vocab"])\n            self.labels = ckpt["labels"]\n            self.max_len = ckpt["max_len"]\n            self.model = JEPAIntentModel(len(self.vocab.stoi), len(self.labels))\n            self.model.load_state_dict(ckpt["state_dict"])\n            self.model.eval()\n        elif self.kind == "openjev":\n            from transformers import AutoModelForSequenceClassification, AutoTokenizer\n            self.tokenizer = AutoTokenizer.from_pretrained(self.root / "openjev", local_files_only=True)\n            self.model = AutoModelForSequenceClassification.from_pretrained(self.root / "openjev", local_files_only=True).eval()\n        else:\n            raise ValueError(f"Unknown model: {self.kind}")\n\n    def relation_scores(self, text, hypotheses):\n        import torch\n        template = getattr(self.model.config, "nli_template", None)\n        if not template:\n            raise ValueError("OpenJev artifact must define its pretrained nli_template")\n        encoded = self.tokenizer([template.format(premise=text, hypothesis=h) for h in hypotheses], return_tensors="pt", padding=True, truncation=True, max_length=512)\n        with torch.inference_mode():\n            probs = self.model(**encoded).logits.softmax(-1)\n        mapping = {str(k).lower(): int(v) for k, v in self.model.config.label2id.items()}\n        ent = mapping.get("entailment")\n        if ent is None:\n            raise ValueError("OpenJev artifact is missing the entailment label mapping")\n        return probs[:, ent].cpu().numpy()\n\n    def predict(self, text):\n        if text in self._cache:\n            value = self._cache[text]\n            return decode("unsupported|none",value["confidence"]) if value["confidence"] < self.threshold else value\n        if self.kind in ("tfidf", "minilm"):\n            x = [text] if self.kind == "tfidf" else self.encoder.encode([text], show_progress_bar=False)\n            p = self.model.predict_proba(x)[0]\n            index = p.argmax()\n            value = decode(str(self.model.classes_[index]), p[index])\n        elif self.kind == "jepa":\n            import torch\n            if len(text) > self.max_len:\n                return decode("unsupported|none", 0.0)\n            with torch.inference_mode():\n                p = self.model.classifier(self.model.encode(torch.tensor([self.vocab.encode(text, self.max_len)]))).softmax(-1)[0]\n            value = decode(self.labels[int(p.argmax())], float(p.max()))\n        else:\n            hs = ["La orden pide clasificar por calidad.", "La orden pide clasificar por color.",\n                  "La orden pide detener la clasificacion.", "La orden pide continuar clasificando.",\n                  "La orden pide cancelar la orden.", "La orden solicita una accion no admitida."]\n            p = self.relation_scores(text, hs)\n            index = p.argmax()\n            value = decode(LABELS[index], p[index])\n        if len(self._cache) >= 4096:\n            self._cache.clear()\n        self._cache[text] = value\n        return decode("unsupported|none",value["confidence"]) if value["confidence"] < self.threshold else value\n', 'roboqc/industrial/training.py': '"""Colab training and evaluation helpers. No training runs at import time."""\n\nimport copy\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .models import IntentBundle\nfrom .voice import canonical, understand\n\n\ndef load_records(path):\n    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]\n\n\ndef target(record):\n    return record["intent"] + "|" + record["mode"]\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    import torch\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef export_minilm_onnx(output, sample):\n    """Export the full sentence encoder (including pooling), then validate INT8."""\n    import torch\n    import joblib\n    import onnxruntime as ort\n    from onnxruntime.quantization import QuantType, quantize_dynamic\n    from sentence_transformers import SentenceTransformer\n    from .models import ONNXSentenceEncoder\n    out = Path(output)\n    original = SentenceTransformer(str(out / "encoder"),device="cpu",model_kwargs={"attn_implementation":"eager"})\n    original.eval()\n    destination = out / "encoder_int8"\n    destination.mkdir(exist_ok=True)\n    original.tokenizer.save_pretrained(destination)\n    class Encoder(torch.nn.Module):\n        def __init__(self, model):\n            super().__init__()\n            self.model = model\n\n        def forward(self, input_ids, attention_mask):\n            return self.model({"input_ids":input_ids,"attention_mask":attention_mask})["sentence_embedding"]\n    tokens = original.tokenize(sample[:2])\n    fp32_path = destination / "model.onnx"\n    torch.onnx.export(Encoder(original).eval(),(tokens["input_ids"],tokens["attention_mask"]),str(fp32_path),\n        input_names=["input_ids","attention_mask"],output_names=["sentence_embedding"],opset_version=17,dynamo=False,\n        dynamic_axes={"input_ids":{0:"batch",1:"sequence"},"attention_mask":{0:"batch",1:"sequence"},"sentence_embedding":{0:"batch"}})\n    quantize_dynamic(str(fp32_path),str(destination / "model_int8.onnx"),weight_type=QuantType.QInt8,\n                     op_types_to_quantize=["MatMul"],per_channel=True)\n    config = {"model_file":"model_int8.onnx","max_seq_length":original.max_seq_length,\n              "embedding_dim":original.get_sentence_embedding_dimension()}\n    (destination / "onnx_config.json").write_text(json.dumps(config,indent=2),encoding="utf-8")\n    a = original.encode(sample,show_progress_bar=False)\n    fp32 = ort.InferenceSession(str(fp32_path),providers=["CPUExecutionProvider"])\n    reference_tokens = original.tokenizer(sample,padding=True,truncation=True,max_length=original.max_seq_length,return_tensors="np")\n    b = fp32.run(None,{item.name:reference_tokens[item.name].astype(np.int64) for item in fp32.get_inputs()})[0]\n    np.testing.assert_allclose(a,b,atol=1e-4,rtol=1e-3)\n    quantized = ONNXSentenceEncoder(destination)\n    c = quantized.encode(sample)\n    cosine = np.sum(a*c,axis=1)/(np.linalg.norm(a,axis=1)*np.linalg.norm(c,axis=1))\n    head = joblib.load(out / "minilm.joblib")\n    agreement = float(np.mean(head.predict(a) == head.predict(c)))\n    return {"minimum_embedding_cosine":float(cosine.min()),"classification_agreement":agreement,\n            "passed":bool(cosine.min() >= .99 and agreement == 1.0),"backend":"onnxruntime"}\n\n\ndef train_linear(train, validation, output, encoder_name=None, seed=42, epochs=40, patience=5):\n    import joblib\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    from sklearn.pipeline import FeatureUnion, Pipeline\n    from sklearn.linear_model import SGDClassifier\n    from sklearn.metrics import f1_score\n    out = Path(output)\n    name = "minilm" if encoder_name else "tfidf"\n    if encoder_name:\n        import torch\n        from sentence_transformers import SentenceTransformer\n        encoder = SentenceTransformer(encoder_name, device="cuda" if torch.cuda.is_available() else "cpu")\n        encoder.save(str(out / "encoder"))\n        x = encoder.encode([r["text"] for r in train], batch_size=64, show_progress_bar=True)\n        vx = encoder.encode([r["text"] for r in validation], batch_size=64, show_progress_bar=True)\n        np.save(out / "train_embeddings.npy", x)\n        np.save(out / "validation_embeddings.npy", vx)\n        vectorizer = None\n    else:\n        vectorizer = FeatureUnion([\n            ("word", TfidfVectorizer(ngram_range=(1, 2), max_features=30000)),\n            ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=50000)),\n        ])\n        x = vectorizer.fit_transform([r["text"] for r in train])\n        vx = vectorizer.transform([r["text"] for r in validation])\n    y, vy = np.array([target(r) for r in train]), [target(r) for r in validation]\n    model = SGDClassifier(loss="log_loss", random_state=seed, alpha=1e-5)\n    best, stale, history = -1, 0, []\n    for epoch in range(epochs):\n        order = np.random.default_rng(seed + epoch).permutation(len(y))\n        model.partial_fit(x[order], y[order], classes=np.unique(y))\n        score = f1_score(vy, model.predict(vx), average="macro", zero_division=0)\n        history.append({"epoch": epoch + 1, "macro_f1": float(score)})\n        if score > best:\n            best, stale = score, 0\n            best_model = copy.deepcopy(model)\n            artifact = best_model if vectorizer is None else Pipeline([("features", vectorizer), ("classifier", best_model)])\n            joblib.dump(artifact, out / f"{name}.joblib")\n        else:\n            stale += 1\n        (out / f"{name}_history.json").write_text(json.dumps(history), encoding="utf-8")\n        if stale >= patience:\n            break\n    return {"validation_macro_f1": float(best), "threshold": 0.0}\n\n\ndef train_jepa(train, validation, output, seed=42, epochs=30, patience=5):\n    import torch\n    from roboqc.intent.jepa_model import CharVocab, JEPAIntentModel, jepa_loss\n    from sklearn.metrics import f1_score\n    seed_all(seed)\n    out = Path(output)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    labels = sorted({target(r) for r in train})\n    vocab = CharVocab()\n    vocab.build([r["text"] for r in train])\n    max_len = 512\n    if any(len(r["text"]) > max_len for r in train + validation):\n        raise ValueError("JEPA max_len would truncate a labelled instruction")\n    x = torch.tensor([vocab.encode(r["text"], max_len) for r in train])\n    y = torch.tensor([labels.index(target(r)) for r in train])\n    vx = torch.tensor([vocab.encode(r["text"], max_len) for r in validation])\n    vy = [target(r) for r in validation]\n    model = JEPAIntentModel(len(vocab.stoi), len(labels)).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)\n    checkpoint = out / "jepa_checkpoint.pt"\n    first, best, stale, history = 0, -1.0, 0, []\n    if checkpoint.exists():\n        state = torch.load(checkpoint, map_location=device, weights_only=True)\n        model.load_state_dict(state["model"])\n        optimizer.load_state_dict(state["optimizer"])\n        first, best, stale, history = state["epoch"], state["best"], state["stale"], state["history"]\n    for epoch in range(first, epochs):\n        model.train()\n        for ids in torch.randperm(len(x)).split(64):\n            full = x[ids].to(device)\n            masked = full.clone()\n            masked[(torch.rand_like(full.float()) < .15) & (full != 0)] = 2\n            optimizer.zero_grad()\n            pred, target_z, logits = model.forward_train(masked, full)\n            loss, _ = jepa_loss(pred, target_z, logits, y[ids].to(device))\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            optimizer.step()\n            model.update_target_encoder()\n        model.eval()\n        predictions = []\n        with torch.inference_mode():\n            for batch in vx.split(128):\n                predictions += [labels[i] for i in model.classifier(model.encode(batch.to(device))).argmax(-1).cpu().tolist()]\n        score = float(f1_score(vy, predictions, average="macro", zero_division=0))\n        history.append({"epoch": epoch + 1, "macro_f1": score})\n        if score > best:\n            best, stale = score, 0\n            torch.save({"state_dict": model.cpu().state_dict(), "vocab": vocab.to_dict(), "labels": labels, "max_len": max_len}, out / "jepa.pt")\n            model.to(device)\n        else:\n            stale += 1\n        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "epoch": epoch + 1,\n                    "best": best, "stale": stale, "history": history}, checkpoint)\n        (out / "jepa_history.json").write_text(json.dumps(history), encoding="utf-8")\n        if stale >= patience:\n            break\n    return {"validation_macro_f1": best, "threshold": 0.0}\n\n\ndef train_openjev(train, validation, output, epochs=3, seed=42):\n    import torch\n    from datasets import Dataset\n    from peft import LoraConfig, TaskType, get_peft_model\n    from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,\n                              EarlyStoppingCallback, Trainer, TrainingArguments)\n    from sklearn.metrics import f1_score\n    from .dataset import pairs\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a Colab GPU for OpenJev LoRA training.")\n    name, sub = "AlexWortega/openjev", "qwen3.5-0.8b-nli-v5"\n    tokenizer = AutoTokenizer.from_pretrained(name, subfolder=sub)\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    model = AutoModelForSequenceClassification.from_pretrained(name, subfolder=sub, torch_dtype=torch.float32)\n    mapping = {k.lower(): int(v) for k, v in model.config.label2id.items()}\n    if set(mapping) != {"entailment", "contradiction", "neutral"}:\n        raise ValueError(f"Inspect upstream label mapping before training: {mapping}")\n    model.config.pad_token_id = tokenizer.pad_token_id\n    model.config.use_cache = False\n    template = getattr(model.config, "nli_template", None)\n    if not template:\n        raise ValueError("Inspect the pretrained OpenJev nli_template before training")\n    model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16,\n                                           target_modules=["q_proj", "v_proj"], lora_dropout=.05))\n    def prepare(records):\n        rows = [{"premise": p, "hypothesis": h, "labels": mapping[label]} for r in records for p, h, label in pairs(r)]\n        data = Dataset.from_list(rows)\n        return data.map(lambda b: tokenizer([template.format(premise=p, hypothesis=h) for p,h in zip(b["premise"], b["hypothesis"])], truncation=True, max_length=512), batched=True,\n                        remove_columns=["premise", "hypothesis"])\n    out = Path(output)\n    args = TrainingArguments(output_dir=str(out / "openjev_checkpoints"), num_train_epochs=epochs,\n        per_device_train_batch_size=2, per_device_eval_batch_size=4, gradient_accumulation_steps=8,\n        learning_rate=2e-4, eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,\n        metric_for_best_model="macro_f1", save_total_limit=2, seed=seed, report_to="none",\n        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})\n    trainer = Trainer(model=model, args=args, train_dataset=prepare(train), eval_dataset=prepare(validation),\n        processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),\n        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],\n        compute_metrics=lambda p: {"macro_f1": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro", zero_division=0)})\n    checkpoints = list((out / "openjev_checkpoints").glob("checkpoint-*"))\n    trainer.train(resume_from_checkpoint=True if checkpoints else None)\n    merged = trainer.model.merge_and_unload()\n    merged.save_pretrained(out / "openjev", safe_serialization=True)\n    tokenizer.save_pretrained(out / "openjev")\n    return {"threshold": 0.0, "upstream": name, "subfolder": sub}\n\n\ndef evaluate(records, model=None, clear_cache=True):\n    from sklearn.metrics import classification_report, confusion_matrix\n    predictions, timings = [], []\n    if model is not None and clear_cache:\n        model._cache.clear()\n    for record in records:\n        start = time.perf_counter()\n        response = understand(record["text"], model)\n        timings.append((time.perf_counter() - start) * 1000)\n        predictions.append(response)\n    true_status = [r["expected"]["status"] for r in records]\n    pred_status = [p["status"] for p in predictions]\n    true_intents = [(r["expected"]["command"] or {}).get("intent", "unsupported") for r in records]\n    pred_intents = [(p["command"] or {}).get("intent", "unsupported") for p in predictions]\n    def fields(command):\n        return {(r["category"], r["destination"]) for r in (command or {}).get("rules", [])}\n    tp = fp = fn = exact = false_accept = valid_count = 0\n    field_correct = {"category": 0, "destination": 0, "mode": 0}\n    entity_counts = {"category":[0,0,0],"destination":[0,0,0]}\n    for record, prediction in zip(records, predictions):\n        gold = record["expected"]\n        equal = prediction["status"] == gold["status"]\n        if gold["status"] in ("valid", "needs_clarification"):\n            equal &= canonical(prediction["command"]) == canonical(gold["command"])\n        exact += equal\n        false_accept += gold["status"] != "valid" and prediction["status"] == "valid"\n        a, b = fields(gold["command"]), fields(prediction["command"])\n        tp += len(a & b); fp += len(b - a); fn += len(a - b)\n        for field, index in [("category", 0), ("destination", 1)]:\n            gold_values,pred_values = {v[index] for v in a},{v[index] for v in b}\n            field_correct[field] += gold_values == pred_values\n            totals = entity_counts[field]\n            totals[0] += len(gold_values & pred_values)\n            totals[1] += len(pred_values-gold_values)\n            totals[2] += len(gold_values-pred_values)\n        field_correct["mode"] += (gold["command"] or {}).get("mode") == (prediction["command"] or {}).get("mode")\n        valid_count += gold["status"] == "valid"\n    labels = sorted(set(true_status + pred_status))\n    report = {"n": len(records), "complete_command_accuracy": exact / max(1, len(records)),\n        "false_acceptance_rate": false_accept / max(1, len(records) - valid_count),\n        "relation_f1": 2 * tp / max(1, 2 * tp + fp + fn),\n        "entity_accuracy": {k: v / max(1, len(records)) for k, v in field_correct.items()},\n        "entity_f1": {k: 2*v[0]/max(1,2*v[0]+v[1]+v[2]) for k,v in entity_counts.items()},\n        "intent": classification_report(true_intents, pred_intents, output_dict=True, zero_division=0),\n        "status": classification_report(true_status, pred_status, output_dict=True, zero_division=0),\n        "confusion_labels": labels, "confusion_matrix": confusion_matrix(true_status, pred_status, labels=labels).tolist(),\n        "latency_ms": {"p50": float(np.percentile(timings, 50)), "p95": float(np.percentile(timings, 95))}}\n    if model is not None:\n        raw = [model.predict(r["text"]) for r in records]\n        report["raw_model_intent"] = classification_report([r["intent"] for r in records], [r["intent"] for r in raw], output_dict=True, zero_division=0)\n        report["raw_model_mode_accuracy"] = float(np.mean([a["mode"] == b["mode"] for a,b in zip(records, raw)]))\n    return report, predictions\n\n\ndef select_and_calibrate(output, validation):\n    """Only validation chooses thresholds and model; test is never consulted."""\n    out = Path(output)\n    manifest = json.loads((out / "manifest.json").read_text())\n    ranks = []\n    for kind in manifest["models"]:\n        model = IntentBundle(out, kind)\n        model.predict(validation[0]["text"])\n        start = time.perf_counter()\n        for record in validation[1:17]:\n            model.predict(record["text"])\n        uncached_latency = (time.perf_counter()-start)*1000 / max(1,len(validation[1:17]))\n        best = None\n        for threshold in [0.0, .5, .7, .85, .95]:\n            model.threshold = threshold\n            report, _ = evaluate(validation, model, clear_cache=False)\n            rank = (report["false_acceptance_rate"], -report["complete_command_accuracy"], -report["raw_model_intent"]["macro avg"]["f1-score"],uncached_latency)\n            if best is None or rank < best[0]:\n                best = rank, threshold, report\n        manifest["models"][kind].update(threshold=best[1], validation=best[2],uncached_cpu_ms=uncached_latency)\n        ranks.append((best[0], kind))\n        del model\n    manifest["selected"] = min(ranks)[1]\n    (out / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    return manifest\n', 'roboqc/intent/jepa_model.py': '"""Compact JEPA-inspired text representation model for Spanish intent classification.\n\nA small character-level encoder predicts a target (full sentence) latent\nrepresentation from a context (partially masked sentence) representation.\nAn EMA target encoder and a variance regulariser are used as anti-collapse\nmeasures. A linear classification head is trained jointly.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Dict, List, Optional\n\ntry:\n    import torch\n    import torch.nn as nn\n    import torch.nn.functional as F\n    _TORCH = True\nexcept Exception:  # pragma: no cover\n    _TORCH = False\n    torch = None\n    nn = None\n    F = None\n\n\ndef is_available() -> bool:\n    return _TORCH\n\n\nPAD_TOKEN = "<pad>"\nUNK_TOKEN = "<unk>"\nMASK_TOKEN = "<mask>"\n\n\nclass CharVocab:\n    def __init__(self):\n        self.stoi: Dict[str, int] = {PAD_TOKEN: 0, UNK_TOKEN: 1, MASK_TOKEN: 2}\n        self.itos: List[str] = [PAD_TOKEN, UNK_TOKEN, MASK_TOKEN]\n\n    def build(self, texts: List[str]) -> None:\n        chars = set()\n        for t in texts:\n            for ch in t.lower():\n                chars.add(ch)\n        for ch in sorted(chars):\n            if ch not in self.stoi:\n                self.stoi[ch] = len(self.itos)\n                self.itos.append(ch)\n\n    def encode(self, text: str, max_len: int = 128) -> List[int]:\n        ids = [self.stoi.get(ch.lower(), self.stoi[UNK_TOKEN]) for ch in text.lower()]\n        ids = ids[:max_len]\n        if len(ids) < max_len:\n            ids = ids + [self.stoi[PAD_TOKEN]] * (max_len - len(ids))\n        return ids\n\n    def mask_sequence(self, ids: List[int], mask_prob: float = 0.15) -> List[int]:\n        out = list(ids)\n        for i in range(len(out)):\n            if out[i] in (0,):  # skip pad\n                continue\n            if torch.rand(1).item() < mask_prob:  # type: ignore\n                out[i] = self.stoi[MASK_TOKEN]\n        return out\n\n    def to_dict(self) -> Dict:\n        return {"stoi": self.stoi, "itos": self.itos}\n\n    @classmethod\n    def from_dict(cls, d: Dict) -> "CharVocab":\n        v = cls()\n        v.stoi = dict(d["stoi"])\n        v.itos = list(d["itos"])\n        return v\n\n\nif _TORCH:\n    class TextEncoder(nn.Module):\n        def __init__(self, vocab_size: int, embed_dim: int = 32, hidden: int = 64, latent: int = 64):\n            super().__init__()\n            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)\n            self.conv1 = nn.Conv1d(embed_dim, hidden, kernel_size=3, padding=1)\n            self.conv2 = nn.Conv1d(hidden, hidden, kernel_size=3, padding=1)\n            self.proj = nn.Linear(hidden, latent)\n\n        def forward(self, x):\n            # x: (B, L) long\n            e = self.embedding(x).transpose(1, 2)  # (B, E, L)\n            h = torch.relu(self.conv1(e))\n            h = torch.relu(self.conv2(h))\n            h = h.max(dim=2).values  # (B, hidden)\n            z = self.proj(h)\n            return z\n\n    class JEPAIntentModel(nn.Module):\n        def __init__(self, vocab_size: int, num_classes: int, embed_dim: int = 32,\n                     hidden: int = 64, latent: int = 64):\n            super().__init__()\n            self.embed_dim = embed_dim\n            self.hidden = hidden\n            self.latent = latent\n            self.context_encoder = TextEncoder(vocab_size, embed_dim, hidden, latent)\n            self.target_encoder = TextEncoder(vocab_size, embed_dim, hidden, latent)\n            self.predictor = nn.Sequential(\n                nn.Linear(latent, latent),\n                nn.ReLU(),\n                nn.Linear(latent, latent),\n            )\n            self.classifier = nn.Linear(latent, num_classes)\n            # init target encoder = context encoder\n            self.target_encoder.load_state_dict(self.context_encoder.state_dict())\n            for p in self.target_encoder.parameters():\n                p.requires_grad = False\n\n        def encode(self, x):\n            return self.context_encoder(x)\n\n        def forward_train(self, context_ids, target_ids):\n            z_c = self.context_encoder(context_ids)\n            with torch.no_grad():\n                z_t = self.target_encoder(target_ids)\n            z_pred = self.predictor(z_c)\n            logits = self.classifier(z_c)\n            return z_pred, z_t, logits\n\n        @torch.no_grad()\n        def update_target_encoder(self, tau: float = 0.996):\n            for ps, pt in zip(self.context_encoder.parameters(), self.target_encoder.parameters()):\n                pt.data.mul_(tau).add_(ps.data, alpha=1 - tau)\n\n    def variance_loss(z, gamma: float = 1.0):\n        # VICReg-style variance: keep std above gamma\n        std = torch.sqrt(z.var(dim=0) + 1e-4)\n        return torch.mean(torch.relu(gamma - std))\n\n    def jepa_loss(z_pred, z_t, logits, labels, lambda_var: float = 0.1):\n        pred_loss = F.mse_loss(z_pred, z_t)\n        cls_loss = F.cross_entropy(logits, labels)\n        var = variance_loss(z_pred) + variance_loss(z_t.detach())\n        total = pred_loss + cls_loss + lambda_var * var\n        return total, {"pred": pred_loss.item(), "cls": cls_loss.item(), "var": var.item()}\nelse:\n    TextEncoder = None\n    JEPAIntentModel = None\n    def variance_loss(*a, **k):\n        raise RuntimeError("torch no disponible")\n    def jepa_loss(*a, **k):\n        raise RuntimeError("torch no disponible")\n', 'roboqc/industrial/vision_data.py': '"""Real industrial annotation import and grouped splits, called only by Colab."""\n\nimport hashlib\nimport json\nimport random\nimport shutil\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nfrom .vision_inference import CLASSES, QUALITY\n\n\ndef _safe(root, relative):\n    path = (Path(root) / relative).resolve()\n    if not path.is_relative_to(Path(root).resolve()):\n        raise ValueError(f"Image path outside dataset: {relative}")\n    return path\n\n\ndef import_coco(root, annotation_file):\n    from pycocotools import mask as mask_utils\n    root = Path(root)\n    data = json.loads(Path(annotation_file).read_text(encoding="utf-8"))\n    categories = {c["id"]: c["name"] for c in data["categories"]}\n    if set(categories.values()) - set(CLASSES):\n        raise ValueError(f"Use these category names: {CLASSES}")\n    grouped = {}\n    for ann in data["annotations"]:\n        if ann.get("iscrowd", 0):\n            raise ValueError("Crowd regions cannot be used as individual graspable instances")\n        grouped.setdefault(ann["image_id"], []).append(ann)\n    records = []\n    for image in data["images"]:\n        path = _safe(root, image["file_name"])\n        frame = cv2.imread(str(path))\n        if frame is None:\n            raise ValueError(f"Unreadable image: {path}")\n        h, w = frame.shape[:2]\n        if [h, w] != [image["height"], image["width"]]:\n            raise ValueError(f"Annotation dimensions differ: {path}")\n        instances = []\n        for ann in grouped.get(image["id"], []):\n            segmentation = ann["segmentation"]\n            if isinstance(segmentation, list):\n                rle = mask_utils.merge(mask_utils.frPyObjects(segmentation, h, w))\n            elif isinstance(segmentation.get("counts"), list):\n                rle = mask_utils.frPyObjects(segmentation, h, w)\n            else:\n                rle = segmentation\n            mask = mask_utils.decode(rle).astype(np.uint8)\n            if mask.shape != (h,w) or not mask.any():\n                raise ValueError(f"Invalid mask in annotation {ann[\'id\']}")\n            attrs = ann.get("attributes", {})\n            quality = attrs.get("quality", "unknown")\n            color = attrs.get("color")\n            if quality not in QUALITY + ["unknown"] or color not in [None,"red","green","blue","unknown"]:\n                raise ValueError("Unsupported instance attributes")\n            instances.append({"class": categories[ann["category_id"]], "mask": mask,\n                              "quality": quality, "color": color})\n        session = image.get("session_id")\n        if session is None:\n            raise ValueError("Each COCO image needs session_id to avoid capture-session leakage")\n        records.append({"path": str(path), "session": str(session), "instances": instances})\n    return records\n\n\ndef import_yolo(root, yaml_file, sessions_file, attributes_file=None):\n    import yaml\n    root = Path(root)\n    data = yaml.safe_load(Path(yaml_file).read_text(encoding="utf-8"))\n    names = data["names"]\n    names = [names[k] for k in sorted(names)] if isinstance(names, dict) else names\n    if set(names) - set(CLASSES):\n        raise ValueError(f"Use these categories: {CLASSES}")\n    sessions = json.loads(Path(sessions_file).read_text(encoding="utf-8"))\n    attributes = json.loads(Path(attributes_file).read_text(encoding="utf-8")) if attributes_file else {}\n    records = []\n    # Imported train/val/test folders are regrouped by session; no augmentations yet.\n    for path in sorted(root.rglob("*")):\n        if path.suffix.lower() not in [".jpg", ".jpeg", ".png", ".bmp"] or "images" not in path.parts:\n            continue\n        relative = path.relative_to(root).as_posix()\n        if relative not in sessions:\n            raise ValueError(f"Missing capture session for {relative}")\n        parts = list(path.parts)\n        parts[max(i for i,p in enumerate(parts) if p == "images")] = "labels"\n        label = Path(*parts).with_suffix(".txt")\n        if not label.exists():\n            raise ValueError(f"Missing label file, including for empty images: {label}")\n        frame = cv2.imread(str(path))\n        if frame is None:\n            raise ValueError(f"Unreadable image: {path}")\n        h, w = frame.shape[:2]\n        instances = []\n        for i, line in enumerate(label.read_text().splitlines()):\n            values = np.array([float(v) for v in line.split()])\n            if len(values) < 7 or (len(values)-1) % 2 or not np.isfinite(values).all() or np.any(values[1:] < 0) or np.any(values[1:] > 1):\n                raise ValueError(f"Expected normalized segmentation polygons: {label}:{i+1}")\n            class_id = int(values[0])\n            if values[0] != class_id or not 0 <= class_id < len(names):\n                raise ValueError(f"Invalid class ID: {label}:{i+1}")\n            polygon = np.rint(values[1:].reshape(-1,2) * [w,h]).astype(np.int32)\n            mask = np.zeros((h,w), np.uint8)\n            cv2.fillPoly(mask, [polygon], 1)\n            attrs = attributes.get(relative, [{}] * (i+1))\n            attrs = attrs[i] if i < len(attrs) else {}\n            instances.append({"class": names[class_id], "mask": mask, "quality": attrs.get("quality", "unknown"), "color": attrs.get("color")})\n        records.append({"path": str(path), "session": str(sessions[relative]), "instances": instances})\n    return records\n\n\ndef prepare(records, output, definitions=None, seed=42):\n    import yaml\n    definitions = definitions or {}\n    if not records:\n        raise ValueError("No real annotated images found. Import a COCO or YOLO dataset first.")\n    groups = sorted({r["session"] for r in records})\n    if len(groups) < 3:\n        raise ValueError("Capture at least three independent sessions for train/validation/test")\n    random.Random(seed).shuffle(groups)\n    n_val = max(1, round(len(groups)*.15)); n_test = max(1, round(len(groups)*.15))\n    split_of = {g: ("val" if i < n_val else "test" if i < n_val+n_test else "train") for i,g in enumerate(groups)}\n    out = Path(output)\n    out.mkdir(parents=True, exist_ok=False)\n    hashes, manifest, losses = {}, [], []\n    for record in records:\n        path = Path(record["path"])\n        split = split_of[record["session"]]\n        frame = cv2.imread(str(path))\n        digest = hashlib.sha256(frame.tobytes()).hexdigest()\n        if digest in hashes:\n            raise ValueError(f"Duplicate image {path} (already in {hashes[digest]}); remove before splitting")\n        hashes[digest] = str(path)\n        h,w = frame.shape[:2]\n        stem = digest[:20]\n        for folder in ["images", "labels", "masks"]:\n            (out / folder / split).mkdir(parents=True, exist_ok=True)\n        cv2.imwrite(str(out / "images" / split / f"{stem}.png"), frame)\n        polygons, instances = [], []\n        for i, instance in enumerate(record["instances"]):\n            mask = instance["mask"]\n            if not mask.any():\n                raise ValueError("Empty mask")\n            contours, hierarchy = cv2.findContours(mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)\n            external = [c for j,c in enumerate(contours) if hierarchy[0,j,3] < 0]\n            contour = max(external, key=cv2.contourArea).reshape(-1,2)\n            if len(contour) < 3:\n                raise ValueError("Degenerate contour")\n            if len(external) > 1 or np.any(hierarchy[0,:,3] >= 0):\n                losses.append({"image": stem, "instance": i, "reason": "YOLO polygon drops holes/disconnected regions; original PNG retained"})\n            coordinates = (contour / [w,h]).clip(0,1).reshape(-1)\n            polygons.append(str(CLASSES.index(instance["class"])) + " " + " ".join(f"{v:.6f}" for v in coordinates))\n            mask_relative = f"masks/{split}/{stem}_{i}.png"\n            cv2.imwrite(str(out / mask_relative), mask*255)\n            instances.append({k:v for k,v in instance.items() if k != "mask"} | {"mask_path": mask_relative})\n        (out / "labels" / split / f"{stem}.txt").write_text("\\n".join(polygons), encoding="utf-8")\n        manifest.append({"image": f"images/{split}/{stem}.png", "split": split, "session": record["session"], "instances": instances})\n    for split in ["train","val","test"]:\n        present = {i["class"] for r in manifest if r["split"] == split for i in r["instances"]}\n        if present != set(CLASSES):\n            raise ValueError(f"{split} needs real examples of all classes; missing {set(CLASSES)-present}. Add sessions or change seed.")\n    (out / "data.yaml").write_text(yaml.safe_dump({"path": str(out.resolve()), "train": "images/train", "val": "images/val", "test": "images/test", "names": CLASSES}), encoding="utf-8")\n    (out / "instances.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    (out / "quality_definitions.json").write_text(json.dumps(definitions, indent=2), encoding="utf-8")\n    (out / "conversion_warnings.json").write_text(json.dumps(losses, indent=2), encoding="utf-8")\n    return manifest\n\n\ndef augment_training(output, copies=1, seed=42):\n    """Matched image/mask transforms, train only; never alter hue labels."""\n    out = Path(output)\n    records = json.loads((out / "instances.json").read_text())\n    rng = np.random.default_rng(seed)\n    added = []\n    for record in records:\n        if record["split"] != "train" or record.get("augmentation_parent"):\n            continue\n        original = cv2.imread(str(out / record["image"]))\n        masks = [cv2.imread(str(out / i["mask_path"]),0) for i in record["instances"]]\n        for index in range(copies):\n            turns = int(rng.integers(1,4))\n            frame = np.rot90(original,turns).copy()\n            transformed = [np.rot90(m,turns).copy() for m in masks]\n            if rng.random() < .5:\n                frame = cv2.flip(frame,1)\n                transformed = [cv2.flip(m,1) for m in transformed]\n            frame = np.clip(frame.astype(np.float32)*rng.uniform(.85,1.15),0,255).astype(np.uint8)\n            stem = Path(record["image"]).stem + f"_aug{index}"\n            image_path = f"images/train/{stem}.png"\n            if (out / image_path).exists():\n                raise FileExistsError("Augmentations already exist; use a new run")\n            cv2.imwrite(str(out / image_path),frame)\n            h,w = frame.shape[:2]\n            annotations, lines = [], []\n            for j,(instance,mask) in enumerate(zip(record["instances"],transformed)):\n                contours,_ = cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)\n                polygon = max(contours,key=cv2.contourArea).reshape(-1,2) / [w,h]\n                lines.append(str(CLASSES.index(instance["class"])) + " " + " ".join(f"{v:.6f}" for v in polygon.reshape(-1)))\n                mask_path = f"masks/train/{stem}_{j}.png"\n                cv2.imwrite(str(out / mask_path),mask)\n                annotations.append({**instance,"mask_path":mask_path})\n            (out / f"labels/train/{stem}.txt").write_text("\\n".join(lines))\n            added.append({**record,"image":image_path,"instances":annotations,"augmentation_parent":record["image"]})\n    (out / "instances.json").write_text(json.dumps(records+added,indent=2))\n    return records+added\n', 'roboqc/industrial/vision_training.py': '"""Manual Colab training/export helpers for real instance annotations."""\n\nimport json\nimport shutil\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nfrom .vision_inference import CLASSES, QUALITY, quality_crop\n\n\ndef train_segmentation(data, output, epochs=80, seed=42, imgsz=640, resume=False):\n    import torch\n    import yaml\n    from ultralytics import YOLO\n    data, out = Path(data), Path(output)\n    config = yaml.safe_load((data / "data.yaml").read_text())\n    config["path"] = str(data.resolve())\n    local_yaml = out / "data_resolved.yaml"\n    local_yaml.write_text(yaml.safe_dump(config), encoding="utf-8")\n    checkpoint = out / "segmentation_train/weights/last.pt"\n    model = YOLO(str(checkpoint) if resume and checkpoint.exists() else "yolo11n-seg.pt")\n    model.train(data=str(local_yaml), project=str(out), name="segmentation_train", exist_ok=resume,\n        epochs=epochs, imgsz=imgsz, batch=8 if torch.cuda.is_available() else 2,\n        device=0 if torch.cuda.is_available() else "cpu", workers=2, seed=seed,\n        patience=15, resume=bool(resume and checkpoint.exists()),\n        degrees=180, translate=.10, scale=.25, fliplr=.5, flipud=.5,\n        hsv_h=0, hsv_s=.15, hsv_v=.25, mosaic=.3, close_mosaic=10,\n        mixup=0, overlap_mask=False)\n    best = YOLO(str(out / "segmentation_train/weights/best.pt"))\n    metrics = best.val(data=str(local_yaml), split="val", device="cpu")\n    shutil.copy2(out / "segmentation_train/weights/best.pt", out / "segmentation.pt")\n    exported = best.export(format="onnx", imgsz=imgsz, dynamic=False, simplify=False, opset=17, device="cpu")\n    shutil.copy2(exported, out / "segmentation.onnx")\n    summary = {k: float(v) for k,v in metrics.results_dict.items()}\n    (out / "segmentation_validation.json").write_text(json.dumps(summary, indent=2))\n    manifest = {"schema_version": "industrial-vision-1.0", "classes": CLASSES, "imgsz": imgsz,\n                "quality_enabled": False, "quality_threshold": .8, "seed": seed,\n                "export_verified": False,\n                "geometry": "planar estimates only; grasp candidates unvalidated"}\n    (out / "vision_manifest.json").write_text(json.dumps(manifest, indent=2))\n    # Compare actual postprocessed CPU outputs, not just whether export returned a path.\n    from .vision_inference import Segmenter\n    original, optimized = Segmenter(out, False), Segmenter(out, True)\n    differences = []\n    for path in sorted((data / "images/val").glob("*.png"))[:10]:\n        frame = cv2.imread(str(path))\n        a,b = original.predict(frame), optimized.predict(frame)\n        if len(a) != len(b):\n            raise AssertionError(f"Export changes instance count: {path}")\n        # Match by mask IoU to avoid relying on backend output order.\n        from scipy.optimize import linear_sum_assignment\n        if not a:\n            continue\n        overlaps = np.array([[np.logical_and(x["mask"],y["mask"]).sum()/max(1,np.logical_or(x["mask"],y["mask"]).sum()) for y in b] for x in a])\n        rows, cols = linear_sum_assignment(-overlaps)\n        for i,j in zip(rows,cols):\n            if a[i]["type"] != b[j]["type"] or overlaps[i,j] < .95:\n                raise AssertionError(f"Export changes class/mask: {path}")\n            differences.append(float(overlaps[i,j]))\n    (out / "export_equivalence.json").write_text(json.dumps({"mask_ious": differences, "passed": True}, indent=2))\n    manifest["export_verified"] = True\n    (out / "vision_manifest.json").write_text(json.dumps(manifest, indent=2))\n    return summary\n\n\ndef train_quality(data, output, epochs=20, patience=4, seed=42):\n    import torch\n    from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights\n    data, out = Path(data), Path(output)\n    definitions = json.loads((data / "quality_definitions.json").read_text())\n    records = json.loads((data / "instances.json").read_text())\n    examples = {split: [(r,i) for r in records if r["split"] == split for i in r["instances"] if i["quality"] in QUALITY] for split in ("train","val","test")}\n    if not all(definitions.get(k) for k in QUALITY) or any(set(i["quality"] for r,i in rows) != set(QUALITY) for rows in examples.values()):\n        (out / "quality_pending.json").write_text(json.dumps({"status": "pending", "reason": "Define all three observable classes and annotate every split"}))\n        return {"status": "pending"}\n    torch.manual_seed(seed)\n    def tensors(rows):\n        x, y = [], []\n        for record, instance in rows:\n            frame = cv2.imread(str(data / record["image"]))\n            mask = cv2.imread(str(data / instance["mask_path"]), 0) > 0\n            rgb = quality_crop(frame,mask).astype(np.float32)/255\n            x.append(((rgb-[.485,.456,.406])/[.229,.224,.225]).transpose(2,0,1))\n            y.append(QUALITY.index(instance["quality"]))\n        return torch.tensor(np.array(x), dtype=torch.float32), torch.tensor(y)\n    tx,ty = tensors(examples["train"])\n    vx,vy = tensors(examples["val"])\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)\n    model.classifier[-1] = torch.nn.Linear(model.classifier[-1].in_features,3)\n    model.to(device)\n    opt = torch.optim.AdamW(model.parameters(),lr=1e-4)\n    first,best,stale = 0,float("inf"),0\n    checkpoint = out / "quality_checkpoint.pt"\n    if checkpoint.exists():\n        state = torch.load(checkpoint,map_location=device,weights_only=True)\n        model.load_state_dict(state["model"]); opt.load_state_dict(state["optimizer"])\n        first,best,stale = state["epoch"],state["best"],state["stale"]\n    for epoch in range(first,epochs):\n        model.train()\n        for batch in torch.randperm(len(tx)).split(32):\n            opt.zero_grad()\n            loss = torch.nn.functional.cross_entropy(model(tx[batch].to(device)),ty[batch].to(device))\n            loss.backward(); opt.step()\n        model.eval()\n        with torch.inference_mode():\n            score = sum(float(torch.nn.functional.cross_entropy(model(x.to(device)),y.to(device),reduction="sum")) for x,y in zip(vx.split(32),vy.split(32))) / len(vy)\n        if score < best:\n            best,stale = score,0\n            torch.save(model.state_dict(),out / "quality.pt")\n        else:\n            stale += 1\n        torch.save({"model": model.state_dict(),"optimizer":opt.state_dict(),"epoch":epoch+1,"best":best,"stale":stale},checkpoint)\n        if stale >= patience:\n            break\n    model.load_state_dict(torch.load(out / "quality.pt",map_location=device,weights_only=True))\n    model = model.cpu().eval()\n    sample = vx[:1]\n    torch.onnx.export(model,sample,str(out / "quality.onnx"),input_names=["images"],output_names=["logits"],opset_version=17,dynamo=False)\n    import onnxruntime as ort\n    session = ort.InferenceSession(str(out / "quality.onnx"),providers=["CPUExecutionProvider"])\n    with torch.inference_mode():\n        expected = model(sample).numpy()\n        p = torch.cat([model(x).softmax(-1) for x in vx.split(32)]).numpy()\n    np.testing.assert_allclose(expected,session.run(None,{"images":sample.numpy()})[0],atol=1e-4,rtol=1e-3)\n    threshold = 1.01\n    for candidate in [.5,.7,.8,.9,.95,.99]:\n        accepted = p.max(-1) >= candidate\n        if accepted.any() and float(np.mean(p.argmax(-1)[accepted] != vy.numpy()[accepted])) <= .05:\n            threshold = candidate; break\n    manifest = json.loads((out / "vision_manifest.json").read_text())\n    manifest.update(quality_enabled=True,quality_threshold=threshold,quality_definitions=definitions)\n    (out / "vision_manifest.json").write_text(json.dumps(manifest,indent=2))\n    return {"status": "trained", "threshold": threshold, "validation_loss": best}\n', 'roboqc/industrial/vision_inference.py': '"""CPU segmentation, planar geometry and optional visible-quality classifier.\n\nThis module intentionally imports no GUI or robot library and is embedded in Colab.\n"""\n\nimport json\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nCLASSES = ["bolt", "nut", "washer", "bracket"]\nQUALITY = ["good", "defect_1", "defect_2"]\n\n\ndef mask_geometry(mask, symmetric=False, clearance_px=3):\n    mask = np.asarray(mask, dtype=np.uint8)\n    ys, xs = np.nonzero(mask)\n    if len(xs) < 3:\n        return {"center_px": None, "orientation_deg": None, "grasp_candidates": []}\n    center = [float(xs.mean()), float(ys.mean())]\n    eigenvalues, eigenvectors = np.linalg.eigh(np.cov(np.column_stack([xs, ys]), rowvar=False))\n    orientation = None\n    if not symmetric and eigenvalues[-1] > 1.3 * max(eigenvalues[0], 1e-9):\n        axis = eigenvectors[:, -1]\n        orientation = float(np.degrees(np.arctan2(axis[1], axis[0])) % 180)\n    distance = cv2.distanceTransform(mask, cv2.DIST_L2, 5)\n    _, radius, _, point = cv2.minMaxLoc(distance)\n    candidates = []\n    if radius >= clearance_px:\n        candidates.append({"position_px": list(point), "clearance_px": float(radius),\n                           "orientation_deg": orientation, "validated": False})\n    return {"center_px": center, "orientation_deg": orientation, "grasp_candidates": candidates}\n\n\ndef classify_mask_color(frame, mask, minimum_fraction=.6):\n    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)\n    # Erode edges to reduce background contamination; classify only saturated pixels.\n    interior = cv2.erode(mask.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0\n    values = hsv[interior]\n    if not len(values):\n        return None, 0.0\n    ranges = {"red": ((values[:, 0] <= 8) | (values[:, 0] >= 170)),\n              "green": (values[:, 0] >= 35) & (values[:, 0] <= 85),\n              "blue": (values[:, 0] >= 86) & (values[:, 0] <= 130)}\n    votes = {key: float(np.mean(region & (values[:, 1] >= 70) & (values[:, 2] >= 45))) for key, region in ranges.items()}\n    color = max(votes, key=votes.get)\n    confidence = votes[color]\n    return (color if confidence >= minimum_fraction else None), confidence\n\n\ndef quality_crop(frame, mask):\n    ys, xs = np.nonzero(mask)\n    if not len(xs):\n        raise ValueError("Empty instance mask")\n    patch = np.where(mask[..., None] > 0, frame, 127)[ys.min():ys.max()+1, xs.min():xs.max()+1]\n    return cv2.cvtColor(cv2.resize(patch, (224, 224)), cv2.COLOR_BGR2RGB)\n\n\nclass Segmenter:\n    def __init__(self, bundle, prefer_onnx=True, confidence=.35):\n        from ultralytics import YOLO\n        self.root = Path(bundle)\n        self.metadata = json.loads((self.root / "vision_manifest.json").read_text(encoding="utf-8"))\n        if self.metadata.get("schema_version") != "industrial-vision-1.0":\n            raise ValueError("Unsupported vision artifact schema")\n        self.classes = self.metadata["classes"]\n        filename = "segmentation.onnx" if prefer_onnx and (self.root / "segmentation.onnx").exists() else "segmentation.pt"\n        if filename.endswith(".pt") and self.metadata.get("model_family") == "yoloe":\n            from ultralytics import YOLOE\n            self.model = YOLOE(str(self.root / filename), task="segment")\n        else:\n            self.model = YOLO(str(self.root / filename), task="segment")\n        self.confidence = confidence\n        self.quality_session = None\n        if self.metadata.get("quality_enabled"):\n            import onnxruntime as ort\n            self.quality_session = ort.InferenceSession(str(self.root / "quality.onnx"), providers=["CPUExecutionProvider"])\n\n    def predict(self, frame):\n        # PT otherwise uses minimal rectangular padding, unlike static square ONNX.\n        # YOLOE also defaults to agnostic NMS while YOLO(onnx) does not.\n        prediction = self.model.predict(frame, device="cpu", imgsz=self.metadata.get("imgsz", 640),\n            conf=self.confidence, rect=False, agnostic_nms=self.metadata.get("model_family") == "yoloe",\n            iou=.7, max_det=300, retina_masks=False, augment=False, verbose=False)[0]\n        objects = []\n        if prediction.masks is None:\n            return objects\n        masks = prediction.masks.data.cpu().numpy()\n        height, width = frame.shape[:2]\n        for i, (box, cls, confidence) in enumerate(zip(prediction.boxes.xyxy.cpu().numpy(), prediction.boxes.cls.cpu().numpy(), prediction.boxes.conf.cpu().numpy())):\n            # Ultralytics masks may be in letterboxed coordinates, not frame coordinates.\n            from ultralytics.utils.ops import scale_masks\n            import torch\n            restored = scale_masks(torch.as_tensor(masks[i:i+1, None],dtype=torch.float32), (height, width))[0, 0].numpy()\n            mask = (restored > .5).astype(np.uint8)\n            if not mask.any():\n                continue\n            x1, y1, x2, y2 = box.tolist()\n            if int(cls) != cls or not 0 <= int(cls) < len(self.classes):\n                raise ValueError("Prediction class ID does not match the exported class mapping")\n            object_type = self.classes[int(cls)]\n            color, color_confidence = classify_mask_color(frame, mask)\n            geometry = mask_geometry(mask, symmetric=object_type in ("washer", "nut"))\n            if geometry["center_px"] is None:\n                continue\n            quality, quality_confidence = "unknown", 0.0\n            if self.quality_session is not None:\n                crop = quality_crop(frame, mask).astype(np.float32) / 255\n                x = ((crop - [.485, .456, .406]) / [.229, .224, .225]).transpose(2, 0, 1)[None].astype(np.float32)\n                logits = self.quality_session.run(None, {self.quality_session.get_inputs()[0].name: x})[0][0]\n                p = np.exp(logits - logits.max()); p /= p.sum()\n                quality_confidence = float(p.max())\n                if quality_confidence >= self.metadata.get("quality_threshold", .8):\n                    quality = QUALITY[int(p.argmax())]\n            partial = x1 <= 1 or y1 <= 1 or x2 >= width-1 or y2 >= height-1\n            if partial:\n                quality = "unknown"\n                geometry["grasp_candidates"] = []\n            objects.append({"type": object_type, "confidence": float(confidence), "mask": mask,\n                "bbox": [int(x1), int(y1), int(x2-x1), int(y2-y1)], "color": color,\n                "color_confidence": color_confidence, "quality": quality,\n                "quality_confidence": quality_confidence, "partial": partial, **geometry})\n        return objects\n\n\ndef draw_instances(frame, objects):\n    out = frame.copy()\n    for i, obj in enumerate(objects):\n        mask = obj["mask"] > 0\n        color = np.array([(53*i+60) % 255, (97*i+100) % 255, (131*i+180) % 255])\n        out[mask] = (.55*out[mask] + .45*color).astype(np.uint8)\n        x, y, w, h = obj["bbox"]\n        cv2.rectangle(out, (x, y), (x+w, y+h), color.tolist(), 2)\n        cv2.putText(out, f"{obj[\'type\']} {obj[\'confidence\']:.2f} {obj[\'quality\']}", (x, max(16,y-5)), cv2.FONT_HERSHEY_SIMPLEX, .45, (255,255,255), 1)\n        for point in obj["grasp_candidates"]:\n            cv2.circle(out, tuple(map(int, point["position_px"])), 4, (0,255,255), -1)\n    return out\n', 'roboqc/industrial/yoloe.py': '"""Pretrained YOLOE preparation and unlabelled inspection. Nothing runs on import."""\n\nimport json\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nfrom .vision_inference import CLASSES, Segmenter, draw_instances\n\nDEFAULT_PROMPTS = {"bolt":"bolt", "nut":"metal nut", "washer":"metal washer", "bracket":"metal bracket"}\n\n\nDEMO_IMAGES = [\n    {"filename":"bolts_nuts_washers.jpg","title":"Bolts,Nuts & Washers","author":"Sbrennan83",\n     "source":"https://commons.wikimedia.org/wiki/File:Bolts,Nuts_%26_Washers.jpg",\n     "url":"https://upload.wikimedia.org/wikipedia/commons/e/e0/Bolts%2CNuts_%26_Washers.jpg",\n     "license":"CC BY-SA 4.0","license_url":"https://creativecommons.org/licenses/by-sa/4.0/",\n     "sha1":"b39b133808a4a39e49a04d64aa55c4104f2cb7fe"},\n    {"filename":"angle_bracket.jpg","title":"Bronze angle bracket - 4 x 4 cm - A","author":"Fructibus",\n     "source":"https://commons.wikimedia.org/wiki/File:Bronze_angle_bracket_-_4_x_4_cm_-_A.jpg",\n     "url":"https://upload.wikimedia.org/wikipedia/commons/0/0a/Bronze_angle_bracket_-_4_x_4_cm_-_A.jpg",\n     "license":"CC0 1.0","license_url":"https://creativecommons.org/publicdomain/zero/1.0/",\n     "sha1":"54d9402f7f7c55b3964cfd92631edde51d406e23"},\n    {"filename":"four_gray_brackets.jpg","title":"4 x Gray angle bracket - 12.5 cm - A","author":"Fructibus",\n     "source":"https://commons.wikimedia.org/wiki/File:4_x_Gray_angle_bracket_-_12.5_cm_-_A.jpg",\n     "url":"https://upload.wikimedia.org/wikipedia/commons/b/b4/4_x_Gray_angle_bracket_-_12.5_cm_-_A.jpg",\n     "license":"CC0 1.0","license_url":"https://creativecommons.org/publicdomain/zero/1.0/",\n     "sha1":"0fbc4e465a512ba6830cbd17fe744f49e6c32415",\n     "notes":"Multiple formed angle brackets; related geometry, not the reference\'s exact stamped parts."},\n    {"filename":"amplifier_bracket.jpg","title":"Bracket for amp. (25960984807)","author":"Robbie Sproule",\n     "source":"https://commons.wikimedia.org/wiki/File:Bracket_for_amp._(25960984807).jpg",\n     "url":"https://upload.wikimedia.org/wikipedia/commons/3/34/Bracket_for_amp._%2825960984807%29.jpg",\n     "license":"CC BY 2.0","license_url":"https://creativecommons.org/licenses/by/2.0/",\n     "sha1":"fac7bc1c96c891cc2b412334f2028d79a374b94c",\n     "notes":"Stereo pair of the same formed bracket, not two independent scenes; not an object-count benchmark."},\n    {"filename":"channel_connector.jpg","title":"Hilti Schienen Eckverbinder 90 degrees","author":"Best Tech Nick 25",\n     "source":"https://commons.wikimedia.org/wiki/File:Hilti_Schienen_Eckverbinder_90%C2%B0.jpg",\n     "url":"https://upload.wikimedia.org/wikipedia/commons/a/a2/Hilti_Schienen_Eckverbinder_90%C2%B0.jpg",\n     "license":"CC0 1.0","license_url":"https://creativecommons.org/publicdomain/zero/1.0/",\n     "sha1":"58260c1883db49524791dd84a2042e16b348071d",\n     "notes":"Assembled bracket/channel connection: background-clutter test, not a loose bin-picking scene."},\n]\n\n\ndef download_demo_images(output):\n    """Download credited, unlabelled real photographs; never run inference."""\n    import hashlib\n    from datetime import datetime, timezone\n    from email.utils import parsedate_to_datetime\n    from urllib.error import HTTPError\n    from urllib.request import Request, urlopen\n    out = Path(output)\n    out.mkdir(parents=True,exist_ok=True)\n    available,skipped = [],[]\n    blocked_reason = None\n    requested = False\n    for item in DEMO_IMAGES:\n        path = out / item["filename"]\n        if path.exists():\n            data = path.read_bytes()\n        elif blocked_reason:\n            skipped.append({"filename":item["filename"],"reason":blocked_reason,"source":item["source"]})\n            continue\n        else:\n            if requested:\n                time.sleep(2)  # Pace sequential downloads; do not burst from shared Colab IPs.\n            requested = True\n            print("Downloading demo photo:",item["title"],flush=True)\n            request = Request(item["url"],headers={"User-Agent":"RobotekinDemo/1.0 (educational robotics image preview)"})\n            data,failure = None,""\n            for attempt in range(3):\n                try:\n                    with urlopen(request,timeout=45) as response:\n                        data = response.read(16*1024*1024+1)\n                    break\n                except HTTPError as exc:\n                    failure = f"HTTP {exc.code}: {exc.reason}"\n                    delay = 5 * 2**attempt\n                    retry_after = exc.headers.get("Retry-After") if exc.headers else None\n                    if retry_after:\n                        try:\n                            delay = max(delay,float(retry_after))\n                        except ValueError:\n                            try:\n                                deadline = parsedate_to_datetime(retry_after)\n                                delay = max(delay,(deadline-datetime.now(timezone.utc)).total_seconds())\n                            except (TypeError,ValueError,OverflowError):\n                                pass\n                    if exc.code not in (429,500,502,503,504):\n                        break\n                    if attempt == 2 or delay > 60:\n                        if exc.code == 429 or delay > 60:\n                            blocked_reason = failure + "; further network requests deferred; retry later or upload photos"\n                        break\n                    print(f"{failure}; retry {attempt+1}/2 in {delay:g}s",flush=True)\n                    time.sleep(delay)\n                except OSError as exc:\n                    failure = str(exc)\n                    break\n            if data is None:\n                skipped.append({"filename":item["filename"],"reason":failure,"source":item["source"]})\n                print("WARNING: photo unavailable:",item["filename"],"|",failure,flush=True)\n                continue\n        if len(data) > 16*1024*1024 or hashlib.sha1(data).hexdigest() != item["sha1"]:\n            raise ValueError(f"Demo image checksum mismatch: {path}; existing files are not replaced")\n        if cv2.imdecode(np.frombuffer(data,dtype=np.uint8),cv2.IMREAD_COLOR) is None:\n            raise ValueError(f"Unreadable demo photo: {path}")\n        if not path.exists():\n            with path.open("xb") as handle:\n                handle.write(data)\n        print(item["filename"],"|",item["author"],"|",item["license"])\n        if item.get("notes"):\n            print("  Note:",item["notes"])\n        available.append(item)\n    (out / "sources.json").write_text(json.dumps(DEMO_IMAGES,indent=2),encoding="utf-8")\n    report = {"status":"complete" if not skipped else "partial" if available else "unavailable",\n              "requested":len(DEMO_IMAGES),"available":[i["filename"] for i in available],"skipped":skipped}\n    (out / "download_report.json").write_text(json.dumps(report,indent=2),encoding="utf-8")\n    credits = [f"{i[\'filename\']}: {i[\'title\']} by {i[\'author\']}\\n{i[\'source\']}\\n{i[\'license\']}: {i[\'license_url\']}\\nChanges: none (original bytes).\\n{i.get(\'notes\',\'\')}\\n" for i in available]\n    (out / "ATTRIBUTION.txt").write_text("\\n".join(credits),encoding="utf-8")\n    if not available:\n        raise RuntimeError(f"No demo photos available. Details: {out / \'download_report.json\'}. "\n                           "Set USE_DEMO_IMAGES=False and supply IMAGE_PATH or upload photos.")\n    if skipped:\n        print(f"WARNING: continuing with {len(available)}/{len(DEMO_IMAGES)} demo photos. "\n              f"Skipped images and reasons: {out / \'download_report.json\'}",flush=True)\n    print("Demo photos have no instance labels; accuracy/mAP cannot be measured.")\n    return out\n\n\ndef find_images(path, limit=50):\n    path = Path(path)\n    if not path.exists():\n        raise FileNotFoundError(f"Image path does not exist: {path}")\n    candidates = [path] if path.is_file() else sorted(path.rglob("*"))\n    images = [p for p in candidates if p.suffix.lower() in (".png",".jpg",".jpeg",".bmp",".webp")]\n    if not images:\n        raise ValueError("Provide ordinary part photos or a ZIP of photos; no annotations are needed")\n    return images[:limit] if limit is not None else images\n\n\ndef compare_instances(original, exported, minimum_iou=.95):\n    from scipy.optimize import linear_sum_assignment\n    if len(original) != len(exported):\n        raise AssertionError(f"ONNX export changed the instance count: PyTorch={len(original)}, ONNX={len(exported)}")\n    if not original:\n        return []\n    similarities = np.array([[np.logical_and(a["mask"],b["mask"]).sum()/max(1,np.logical_or(a["mask"],b["mask"]).sum())\n                              if a["type"] == b["type"] else 0.0 for b in exported] for a in original])\n    rows, columns = linear_sum_assignment(-similarities)\n    scores = [float(similarities[i,j]) for i,j in zip(rows,columns)]\n    if min(scores) < minimum_iou:\n        raise AssertionError("ONNX export changed classes or masks; retain the PyTorch bundle or investigate export")\n    return scores\n\n\ndef prepare_bundle(output, images, checkpoint="yoloe-26n-seg.pt", prompts=None, imgsz=640,\n                   confidence=.25, export_onnx=True, seed=42):\n    prompts = DEFAULT_PROMPTS if prompts is None else prompts\n    if set(prompts) != set(CLASSES) or any(not isinstance(v,str) or not v.strip() for v in prompts.values()):\n        raise ValueError(f"Provide one nonempty text prompt for each canonical class: {CLASSES}")\n    texts = [prompts[c].strip() for c in CLASSES]\n    if len(set(texts)) != len(texts):\n        raise ValueError("Prompts must distinguish all four classes")\n    if not images:\n        raise ValueError("Supply at least one ordinary photo for export verification")\n    out = Path(output)\n    out.mkdir(parents=True,exist_ok=False)\n    from ultralytics import YOLOE\n    model = YOLOE(str(checkpoint),task="segment")\n    model.set_classes(texts,model.get_text_pe(texts))\n    names = model.names\n    actual = [names[i] for i in range(len(names))] if isinstance(names,dict) else list(names)\n    if actual != texts:\n        raise ValueError("YOLOE class order does not match the requested prompts")\n    # Embeddings are saved in the checkpoint; local inference never calls set_classes.\n    model.save(str(out / "segmentation.pt"))\n    del model\n    model = YOLOE(str(out / "segmentation.pt"),task="segment")\n    manifest = {"schema_version":"industrial-vision-1.0","model_family":"yoloe",\n        "source_checkpoint":str(checkpoint),"classes":list(CLASSES),"prompts":dict(zip(CLASSES,texts)),\n        "imgsz":imgsz,"confidence":confidence,"seed":seed,"quality_enabled":False,\n        "training_performed":False,"export_verified":False,\n        "geometry":"planar estimates only; grasp candidates unvalidated"}\n    manifest_path = out / "vision_manifest.json"\n    manifest_path.write_text(json.dumps(manifest,indent=2),encoding="utf-8")\n    if export_onnx:\n        exported = model.export(format="onnx",imgsz=imgsz,opset=17,dynamic=False,simplify=False,device="cpu")\n        destination = out / "segmentation.onnx"\n        if Path(exported).resolve() != destination.resolve():\n            shutil.copy2(exported,destination)\n    del model\n    original = Segmenter(out,prefer_onnx=False,confidence=confidence)\n    optimized = Segmenter(out,prefer_onnx=True,confidence=confidence) if export_onnx else None\n    comparison = {"onnx_exported":export_onnx,"images":[],"mask_ious":[],"ground_truth_accuracy":"not_measured",\n        "inference":{"imgsz":imgsz,"confidence":confidence,"rect":False,"agnostic_nms":True,\n                     "iou":.7,"max_det":300,"retina_masks":False,"augment":False},"passed":False}\n    comparison_path = out / "export_equivalence.json"\n    for path in list(images)[:3]:\n        frame = cv2.imread(str(path))\n        if frame is None:\n            raise ValueError(f"Unreadable image: {path}")\n        instances = original.predict(frame)\n        entry = {"path":str(path),"predicted_instances":len(instances),\n                 "pytorch":[{k:obj[k] for k in ("type","confidence","bbox")} for obj in instances]}\n        comparison["images"].append(entry)\n        if optimized is not None:\n            onnx_instances = optimized.predict(frame)\n            entry["onnx_predicted_instances"] = len(onnx_instances)\n            entry["onnx"] = [{k:obj[k] for k in ("type","confidence","bbox")} for obj in onnx_instances]\n            try:\n                comparison["mask_ious"].extend(compare_instances(instances,onnx_instances))\n            except AssertionError as exc:\n                comparison["error"] = str(exc)\n                comparison_path.write_text(json.dumps(comparison,indent=2),encoding="utf-8")\n                raise AssertionError(f"{exc}. Image: {path}. Diagnostics: {comparison_path}. "\n                                     "ONNX remains unverified; set EXPORT_ONNX=False and rerun for a PT-only bundle.") from exc\n    comparison["passed"] = True\n    comparison_path.write_text(json.dumps(comparison,indent=2),encoding="utf-8")\n    manifest["export_verified"] = True\n    manifest_path.write_text(json.dumps(manifest,indent=2),encoding="utf-8")\n    return manifest\n\n\ndef inspect_images(bundle, images, output, confidence=.25, cpu_threads=2, max_visualizations=10):\n    """Export overlays and masks; explicitly abstain from unmeasurable accuracy."""\n    import torch\n    torch.set_num_threads(cpu_threads)\n    out = Path(output)\n    out.mkdir(parents=True,exist_ok=False)\n    if not images:\n        raise ValueError("No input images")\n    model = Segmenter(bundle,confidence=confidence)\n    first = cv2.imread(str(images[0]))\n    if first is None:\n        raise ValueError(f"Unreadable image: {images[0]}")\n    model.predict(first)  # warmup, excluded from latency\n    rows,timings = [],[]\n    for number,path in enumerate(images):\n        frame = cv2.imread(str(path))\n        if frame is None:\n            raise ValueError(f"Unreadable image: {path}")\n        start = time.perf_counter()\n        objects = model.predict(frame)\n        timings.append((time.perf_counter()-start)*1000)\n        instances = []\n        for index,obj in enumerate(objects):\n            mask_file = f"{number:04d}_mask_{index:03d}.png"\n            cv2.imwrite(str(out / mask_file),obj["mask"]*255)\n            instances.append({k:v for k,v in obj.items() if k != "mask"} | {"mask_file":mask_file})\n        overlay = None\n        if number < max_visualizations:\n            overlay = f"{number:04d}_overlay.png"\n            cv2.imwrite(str(out / overlay),draw_instances(frame,objects))\n        rows.append({"image":str(path),"objects":instances,"overlay":overlay,"latency_ms":timings[-1]})\n    report = {"evaluation_mode":"unlabelled","images":len(rows),"device":"cpu",\n        "confidence_threshold":confidence,"cpu_threads":cpu_threads,\n        "latency_ms":{"p50":float(np.percentile(timings,50)),"p95":float(np.percentile(timings,95))},\n        "accuracy":{"status":"not_measured","reason":"No ground-truth instance annotations supplied"},\n        "quality":{"status":"unknown","reason":"Pretrained YOLOE is not a defect classifier"}}\n    (out / "vision_report.json").write_text(json.dumps(report,indent=2),encoding="utf-8")\n    (out / "predictions.json").write_text(json.dumps(rows,indent=2),encoding="utf-8")\n    return report,rows\n'}
for relative, source in SOURCES.items():
    target = RUNTIME / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
sys.path.insert(0, str(RUNTIME))
def package_runtime(folder):
    destination = Path(folder) / 'runtime'
    shutil.copytree(RUNTIME / 'roboqc', destination / 'roboqc', dirs_exist_ok=True)
def local_requirements(folder, packages):
    from importlib.metadata import version, PackageNotFoundError
    rows = []
    for package in packages:
        try:
            # Colab CUDA build suffixes are not portable to Windows CPU wheels.
            rows.append(package + '==' + version(package).split('+')[0])
        except PackageNotFoundError:
            pass
    (Path(folder) / 'requirements-local.txt').write_text('\n'.join(rows) + '\n')
def upload_folder():
    from google.colab import files
    uploaded = files.upload()
    target = RUN / ('uploaded_' + uuid.uuid4().hex[:8])
    target.mkdir()
    for name, blob in uploaded.items():
        path = target / Path(name).name
        path.write_bytes(blob)
        if path.suffix == '.zip':
            with zipfile.ZipFile(path) as archive:
                root = target.resolve()
                for member in archive.infolist():
                    if not (root / member.filename).resolve().is_relative_to(root):
                        raise ValueError('Unsafe ZIP member')
                archive.extractall(target)
    return target
def archive_export(folder):
    package_runtime(folder)
    archive = shutil.make_archive(str(folder), 'zip', folder)
    print('Export:', archive)
    return archive


In [ ]:
source = Path(DATASET_PATH) if DATASET_PATH else upload_folder()
manifests = list(source.rglob('instances.json'))
if len(manifests) != 1: raise ValueError('Choose notebook 04 dataset ZIP/folder')
DATA = manifests[0].parent
assert (DATA / 'data.yaml').exists() and (DATA / 'quality_definitions.json').exists()
OUT = RUN / 'vision_bundle'
OUT.mkdir(exist_ok=bool(RESUME_RUN))
from roboqc.industrial.vision_training import train_segmentation, train_quality
if not GPU: print('CPU detected. Consider EPOCHS=3 for a workflow check; this is not a useful trained model.')
if not (OUT / 'vision_manifest.json').exists() or not json.loads((OUT / 'vision_manifest.json').read_text()).get('export_verified'):
    summary = train_segmentation(DATA,OUT,EPOCHS,SEED,IMGSZ,bool(RESUME_RUN))
    print(summary)
if TRAIN_QUALITY:
    print(train_quality(DATA,OUT,epochs=QUALITY_EPOCHS,seed=SEED))


In [ ]:
import shutil
assert json.loads((OUT / 'vision_manifest.json').read_text()).get('export_verified'), 'Complete export verification before packaging'
shutil.copy2(RUN / 'environment.txt',OUT / 'requirements-colab.txt')
shutil.copy2(DATA / 'quality_definitions.json',OUT / 'quality_definitions.json')
local_requirements(OUT,['ultralytics','onnxruntime','opencv-python','numpy','torch','torchvision'])
print((OUT / 'vision_manifest.json').read_text())
archive = archive_export(OUT)
print('Keep this export alongside, not over, existing models.')
